In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q datasets
import torch, numpy as np, pandas as pd
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification

DRIVE = "/content/drive/MyDrive/financial-news-intelligence"
tokenizer = BertTokenizer.from_pretrained(f"{DRIVE}/models/finbert")
finbert = BertForSequenceClassification.from_pretrained(f"{DRIVE}/models/finbert").to("cuda").eval()
print("GPU:", torch.cuda.get_device_name(0))

nifty = load_dataset("raeidsaqur/NIFTY")
days = pd.concat([nifty[s].to_pandas().assign(split=s) for s in ["train", "valid", "test"]], ignore_index=True)
days["headlines"] = days["news"].str.split("\n").apply(lambda hs: [h.strip() for h in hs if h.strip()])

# Flatten: one row per headline, remembering which day it came from
flat = days[["headlines"]].explode("headlines").dropna().reset_index()   # 'index' = day number
texts = flat["headlines"].tolist()
print("days:", len(days), "| total headlines:", len(texts))

probs = []
with torch.no_grad():
    for i in range(0, len(texts), 256):
        enc = tokenizer(texts[i:i+256], truncation=True, max_length=64, padding=True, return_tensors="pt").to("cuda")
        probs.append(finbert(**enc).logits.softmax(dim=-1).float().cpu().numpy())
        if (i // 256) % 50 == 0:
            print(f"{i}/{len(texts)} headlines")
p = np.vstack(probs)                                    # columns: negative, neutral, positive
flat["p_neg"], flat["p_pos"], flat["pred"] = p[:, 0], p[:, 2], p.argmax(axis=1)

daily = flat.groupby("index").agg(
    mood=("p_pos", lambda s: (s - flat.loc[s.index, "p_neg"]).mean()),
    pos_share=("pred", lambda s: (s == 2).mean()),
    neg_share=("pred", lambda s: (s == 0).mean()),
    n=("pred", "size"))
days = days.join(daily)

out = days[["date", "split", "label", "pct_change", "mood", "pos_share", "neg_share", "n"]]
out.to_csv(f"{DRIVE}/results/nifty_daily_sentiment.csv", index=False)
print(out.head())
print("saved to Drive")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/5.35k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 40.0MB            

...

valid.jsonl:   0%|          | 0.00/2.52M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1477 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/317 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/317 [00:00<?, ? examples/s]

days: 2111 | total headlines: 436564
0/436564 headlines
12800/436564 headlines
25600/436564 headlines
38400/436564 headlines
51200/436564 headlines
64000/436564 headlines
76800/436564 headlines
89600/436564 headlines
102400/436564 headlines
115200/436564 headlines
128000/436564 headlines
140800/436564 headlines
153600/436564 headlines
166400/436564 headlines
179200/436564 headlines
192000/436564 headlines
204800/436564 headlines
217600/436564 headlines
230400/436564 headlines
243200/436564 headlines
256000/436564 headlines
268800/436564 headlines
281600/436564 headlines
294400/436564 headlines
307200/436564 headlines
320000/436564 headlines
332800/436564 headlines
345600/436564 headlines
358400/436564 headlines
371200/436564 headlines
384000/436564 headlines
396800/436564 headlines
409600/436564 headlines
422400/436564 headlines
435200/436564 headlines
        date  split    label  pct_change      mood  pos_share  neg_share    n
0 2010-01-06  train  Neutral      0.0042 -0.021639   0.16

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

d = pd.read_csv(PROJECT_ROOT / "results" / "nifty_daily_sentiment.csv")
print(d.groupby("split")["n"].mean().round(0))                 # headlines per day

# 1. Simple check: does a better mood come before a better next day?
for s in ["train", "valid", "test"]:
    x = d[d.split == s]
    print(s, "correlation(mood, next-day change) =", round(x["mood"].corr(x["pct_change"]), 3))

# 2. Learn on 2010-17, test on later years, vs the always-Neutral baseline
F = ["mood", "pos_share", "neg_share"]
tr = d[d.split == "train"]
for cw in [None, "balanced"]:
    m = LogisticRegression(class_weight=cw, max_iter=1000).fit(tr[F], tr["label"])
    for s in ["valid", "test"]:
        x = d[d.split == s]; p = m.predict(x[F])
        print(f"LR class_weight={cw} {s}: acc {accuracy_score(x.label, p):.3f}  Macro-F1 {f1_score(x.label, p, average='macro'):.3f}")
for s in ["valid", "test"]:
    x = d[d.split == s]; p = ["Neutral"] * len(x)
    print(f"Always-Neutral {s}: acc {accuracy_score(x.label, p):.3f}  Macro-F1 {f1_score(x.label, p, average='macro'):.3f}")